# XGBoost baseline on engineered features

Optimize a single XGBoost model on engineered artifacts with fast sampled CV search, full-fold reranking, and a configurable CV performance estimate.

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_sample_weight

from helper_functions.data_preprocessing import encode_label
from helper_functions.gradient_boosting_baseline import add_candidate
from helper_functions.gradient_boosting_search import (
    attach_rerank_results,
    build_score_interval_frame,
    build_stage1_payload,
    normalize_search_payload,
    summarize_with_ci,
)
from helper_functions.xgboost_ensemble import (
    FoldSamplingConfig,
    XGBoostError,
    build_xgb_model,
    make_fixed_sampled_folds,
    summarize_scores,
)
from helper_functions.xgboost_single_model_search import (
    build_relative_xgb_candidate,
    derive_parameter_ranges,
    limited_folds,
    score_xgb_candidates_parallel,
    score_xgb_on_folds_parallel,
)

## 1. Run configuration

In [ ]:
# Inputs from upstream notebooks.
ENGINEERED_CV_FOLDS     = '../data/tmp/05-engineered-cv-folds.pkl'
ENGINEERED_TRAIN_DATA   = '../data/tmp/05-engineered-train-data.csv'
ENGINEERED_TEST_DATA    = '../data/tmp/05-engineered-test-data.csv'
WINNING_HYPERPARAMETERS = '../data/results/03-winning-hyperparameters.pkl'

# Outputs for this notebook.
SEARCH_RESULTS_FILE      = '../data/results/07-xgboost-search-results.pkl'
CROSS_VALIDATION_SCORES  = '../data/results/07-xgboost-scores.pkl'
RERANK_TOP_K_PARAMS_FILE = '../data/results/07-xgboost-rerank-top-k-params.pkl'
FINAL_SUBMISSION_FILE    = '../data/submission.csv'

# Main run toggles.
RUN_PARAMETER_SEARCH      = True
RUN_FULL_CV_ESTIMATE      = True
USE_BALANCED_CLASS_WEIGHT = True

# Round 1 sampled search controls.
SEARCH_FOLD_LIMIT         = 3
TOP_K_FOR_ENSEMBLE_RANGES = 20

SEARCH_USE_SAMPLING           = True
SEARCH_TRAIN_SAMPLE_FRAC      = 0.2
SEARCH_VALIDATION_SAMPLE_FRAC = 0.2

# Round 2 rerank controls.
TOP_K_FOR_RERANK              = 12
RERANK_FOLD_LIMIT             = None
RERANK_USE_SAMPLING           = False
RERANK_TRAIN_SAMPLE_FRAC      = 1.0
RERANK_VALIDATION_SAMPLE_FRAC = 1.0

# Final CV estimate controls.
ESTIMATE_FOLD_LIMIT             = None
ESTIMATE_USE_SAMPLING           = False
ESTIMATE_TRAIN_SAMPLE_FRAC      = 1.0
ESTIMATE_VALIDATION_SAMPLE_FRAC = 1.0

# Compute controls.
PARALLEL_GPU_IDS = (0, 1)
XGB_N_JOBS       = 2

TARGET_TOTAL_CANDIDATES = 500
RANDOM_SEARCH_SEED      = 315

## 2. Load artifacts

In [ ]:
with open(ENGINEERED_CV_FOLDS, 'rb') as handle:
    engineered_folds = pickle.load(handle)

with open(WINNING_HYPERPARAMETERS, 'rb') as handle:
    winning_hgb_params = pickle.load(handle)

train_df = pd.read_csv(ENGINEERED_TRAIN_DATA)
test_df = pd.read_csv(ENGINEERED_TEST_DATA)

raw_train = pd.read_csv(
    'https://media.githubusercontent.com/media/gperdrizet/fullstack-2605/'
    'refs/heads/main/data/student-health-risk-train.csv'
)
_, label_encoder = encode_label(raw_train['health_condition'])

print(f'Loaded {len(engineered_folds)} engineered folds')
print(f'Engineered train shape: {train_df.shape}')
print(f'Engineered test shape:  {test_df.shape}')
print(f'Winning HGB params from notebook 03: {winning_hgb_params}')

## 3. Candidate generation

In [ ]:
base_xgb_params = {
    'max_depth': int(winning_hgb_params.get('max_depth', 6)),
    'learning_rate': float(winning_hgb_params.get('learning_rate', 0.05)),
    'n_estimators': int(winning_hgb_params.get('max_iter', 320)),
    'subsample': 0.85,
    'colsample_bytree': float(winning_hgb_params.get('max_features', 0.8)),
    'reg_lambda': float(winning_hgb_params.get('l2_regularization', 0.1)),
    'reg_alpha': 0.0,
    'min_child_weight': 1.0,
    'n_jobs': XGB_N_JOBS,
}

relative_candidate_specs = [
    {},
    {'learning_rate_mult': 0.75, 'n_estimators_mult': 1.25, 'max_depth_delta': -1, 'subsample_mult': 0.9, 'colsample_mult': 0.9},
    {'learning_rate_mult': 1.00, 'n_estimators_mult': 1.00, 'max_depth_delta': 0, 'subsample_mult': 1.0, 'colsample_mult': 1.0},
    {'learning_rate_mult': 1.20, 'n_estimators_mult': 0.80, 'max_depth_delta': 1, 'subsample_mult': 1.0, 'colsample_mult': 1.0},
    {'learning_rate_mult': 0.85, 'n_estimators_mult': 1.40, 'max_depth_delta': 0, 'subsample_mult': 0.85, 'colsample_mult': 0.85},
]

wide_search_space = {
    'max_depth': [3, 4, 5, 6, 7, 8, 9],
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08, 0.12],
    'n_estimators': [120, 200, 320, 450, 640, 800],
    'subsample': [0.4, 0.55, 0.7, 0.85, 1.0],
    'colsample_bytree': [0.35, 0.5, 0.65, 0.8, 1.0],
    'reg_lambda': [0.0, 0.05, 0.1, 0.3, 1.0, 3.0],
    'reg_alpha': [0.0, 0.01, 0.05, 0.1, 0.3],
    'min_child_weight': [1.0, 2.0, 4.0, 8.0],
}

parameter_candidates = []
seen_candidates = set()

for spec in relative_candidate_specs:
    parameter_candidates, seen_candidates = add_candidate(
        build_relative_xgb_candidate(base_xgb_params, spec),
        parameter_candidates,
        seen_candidates,
    )

parameter_candidates, seen_candidates = add_candidate(
    dict(base_xgb_params),
    parameter_candidates,
    seen_candidates,
)

rng = np.random.default_rng(RANDOM_SEARCH_SEED)
max_random_attempts = max(2000, TARGET_TOTAL_CANDIDATES * 10)
random_attempts = 0

while len(parameter_candidates) < TARGET_TOTAL_CANDIDATES and random_attempts < max_random_attempts:
    random_attempts += 1

    random_candidate = {
        'max_depth': int(rng.choice(wide_search_space['max_depth'])),
        'learning_rate': float(rng.choice(wide_search_space['learning_rate'])),
        'n_estimators': int(rng.choice(wide_search_space['n_estimators'])),
        'subsample': float(rng.choice(wide_search_space['subsample'])),
        'colsample_bytree': float(rng.choice(wide_search_space['colsample_bytree'])),
        'reg_lambda': float(rng.choice(wide_search_space['reg_lambda'])),
        'reg_alpha': float(rng.choice(wide_search_space['reg_alpha'])),
        'min_child_weight': float(rng.choice(wide_search_space['min_child_weight'])),
        'n_jobs': XGB_N_JOBS,
    }

    parameter_candidates, seen_candidates = add_candidate(
        random_candidate,
        parameter_candidates,
        seen_candidates,
    )

print(f'Generated {len(parameter_candidates)} unique parameter candidates.')

if len(parameter_candidates) < TARGET_TOTAL_CANDIDATES:
    print(
        f'Warning: reached {len(parameter_candidates)} unique candidates below target '
        f'{TARGET_TOTAL_CANDIDATES} after {random_attempts} random attempts.'
    )

## 4. Round 1 sampled search

In [ ]:
if RUN_PARAMETER_SEARCH:

    search_folds = limited_folds(engineered_folds, SEARCH_FOLD_LIMIT)

    search_sampling = FoldSamplingConfig(
        use_sampling=SEARCH_USE_SAMPLING,
        train_sample_fraction=SEARCH_TRAIN_SAMPLE_FRAC,
        validation_sample_fraction=SEARCH_VALIDATION_SAMPLE_FRAC,
        sample_seed=RANDOM_SEARCH_SEED,
    )

    search_eval_folds = make_fixed_sampled_folds(search_folds, search_sampling)

    print(
        f'Round 1: {len(parameter_candidates)} candidates on '
        f'{len(search_folds)}/{len(engineered_folds)} folds (sampling={SEARCH_USE_SAMPLING}, '
        f'gpu_workers={len(PARALLEL_GPU_IDS)}, '
        f'~{len(parameter_candidates) * len(search_eval_folds)} fits total)'
    )

    stage1_rows = score_xgb_candidates_parallel(
        parameter_candidates,
        search_eval_folds,
        seed=RANDOM_SEARCH_SEED,
        gpu_ids=PARALLEL_GPU_IDS,
        use_balanced_sample_weight=USE_BALANCED_CLASS_WEIGHT,
    )

    if not stage1_rows:
        raise ValueError('Stage 1 search returned no candidate rows.')

    best_params_stage1 = stage1_rows[0]['params']
    top_stage1_for_ranges = stage1_rows[:min(TOP_K_FOR_ENSEMBLE_RANGES, len(stage1_rows))]

    parameter_ranges = derive_parameter_ranges(
        top_stage1_for_ranges,
        top_n=min(8, len(top_stage1_for_ranges)),
    )

    search_payload = build_stage1_payload(
        stage1_rows=stage1_rows,
        best_params=best_params_stage1,
        config={
            'search_fold_limit': SEARCH_FOLD_LIMIT,
            'search_use_sampling': SEARCH_USE_SAMPLING,
            'search_train_sample_frac': SEARCH_TRAIN_SAMPLE_FRAC,
            'search_validation_sample_frac': SEARCH_VALIDATION_SAMPLE_FRAC,
            'target_total_candidates': TARGET_TOTAL_CANDIDATES,
            'random_search_seed': RANDOM_SEARCH_SEED,
            'parallel_gpu_ids': PARALLEL_GPU_IDS,
            'xgb_n_jobs': XGB_N_JOBS,
            'use_balanced_class_weight': USE_BALANCED_CLASS_WEIGHT,
            'top_k_for_ensemble_ranges': TOP_K_FOR_ENSEMBLE_RANGES,
        },
    )

    search_payload['parameter_ranges'] = parameter_ranges

    with open(SEARCH_RESULTS_FILE, 'wb') as handle:
        pickle.dump(search_payload, handle)

else:
    with open(SEARCH_RESULTS_FILE, 'rb') as handle:
        search_payload = normalize_search_payload(pickle.load(handle))

    stage1_rows = search_payload['stage1_rows']
    best_params_stage1 = search_payload['best_params_stage1']
    parameter_ranges = search_payload.get('parameter_ranges')

    if not parameter_ranges:
        top_stage1_for_ranges = stage1_rows[:min(TOP_K_FOR_ENSEMBLE_RANGES, len(stage1_rows))]

        parameter_ranges = derive_parameter_ranges(
            top_stage1_for_ranges,
            top_n=min(8, len(top_stage1_for_ranges)),
        )

        search_payload['parameter_ranges'] = parameter_ranges
        search_payload.setdefault('config', {})['top_k_for_ensemble_ranges'] = TOP_K_FOR_ENSEMBLE_RANGES

print(f'Best stage-1 params: {best_params_stage1}')
print(f'Parameter ranges for notebook 07: {parameter_ranges}')

In [ ]:
# Plot candidates sorted by median sampled-search score (best to worst) with a 95% fold interval.
sorted_stage1_rows = sorted(stage1_rows, key=lambda row: row['median'], reverse=True)
median_scores_df = build_score_interval_frame(sorted_stage1_rows)

plt.title('Round 1 sampled-search hyperparameter scores')
plt.plot(list(range(len(median_scores_df))), median_scores_df['median_score'], color='black')
plt.fill_between(
    list(range(len(median_scores_df))),
    median_scores_df['95% CI lower'],
    median_scores_df['95% CI upper'],
    color='lightgray',
    alpha=0.5
)
plt.xlabel('Hyperparameter combination')
plt.ylabel('Median balanced accuracy')
plt.show()

## 5. Round 2 rerank

In [ ]:
rerank_folds = limited_folds(engineered_folds, RERANK_FOLD_LIMIT)

rerank_sampling = FoldSamplingConfig(
    use_sampling=RERANK_USE_SAMPLING,
    train_sample_fraction=RERANK_TRAIN_SAMPLE_FRAC,
    validation_sample_fraction=RERANK_VALIDATION_SAMPLE_FRAC,
    sample_seed=RANDOM_SEARCH_SEED + 7,
)

rerank_eval_folds = make_fixed_sampled_folds(rerank_folds, rerank_sampling)
top_stage1_rows = stage1_rows[:min(TOP_K_FOR_RERANK, len(stage1_rows))]

if not top_stage1_rows:
    raise ValueError('No stage-1 candidates available for rerank.')

print(
    f'Round 2: reranking top {len(top_stage1_rows)} candidates on '
    f'{len(rerank_folds)}/{len(engineered_folds)} folds (sampling={RERANK_USE_SAMPLING}, '
    f'gpu_workers={len(PARALLEL_GPU_IDS)}, '
    f'~{len(top_stage1_rows) * len(rerank_eval_folds)} fits total)'
)

rerank_rows = score_xgb_candidates_parallel(
    [row['params'] for row in top_stage1_rows],
    rerank_eval_folds,
    seed=RANDOM_SEARCH_SEED + 10,
    gpu_ids=PARALLEL_GPU_IDS,
    use_balanced_sample_weight=USE_BALANCED_CLASS_WEIGHT,
)

for rerank_row, row in zip(rerank_rows, top_stage1_rows):
    rerank_row['stage1_median'] = row['median']

rerank_rows = sorted(rerank_rows, key=lambda row: row['median'], reverse=True)
best_params = rerank_rows[0]['params']

top_rerank_param_sets = [
    row['params'] for row in rerank_rows[:min(TOP_K_FOR_RERANK, len(rerank_rows))]
]

os.makedirs(os.path.dirname(RERANK_TOP_K_PARAMS_FILE), exist_ok=True)

with open(RERANK_TOP_K_PARAMS_FILE, 'wb') as handle:
    pickle.dump(top_rerank_param_sets, handle)

search_payload = attach_rerank_results(
    search_payload,
    rerank_rows=rerank_rows,
    best_params_selected=best_params,
    config_updates={
        'top_k_for_rerank': TOP_K_FOR_RERANK,
        'rerank_fold_limit': RERANK_FOLD_LIMIT,
        'rerank_use_sampling': RERANK_USE_SAMPLING,
        'rerank_top_k_params_file': RERANK_TOP_K_PARAMS_FILE,
    },
)

search_payload['top_rerank_param_sets'] = top_rerank_param_sets

with open(SEARCH_RESULTS_FILE, 'wb') as handle:
    pickle.dump(search_payload, handle)

print(f'Best params used for final training: {best_params}')
print(f'Saved top rerank parameter sets to {RERANK_TOP_K_PARAMS_FILE}')

In [ ]:
# Plot candidate rank (x-axis) versus median rerank score with 95% fold interval.
median_scores_df = build_score_interval_frame(rerank_rows)

plt.title('Round 2 reranked hyperparameter scores')
plt.plot(list(range(len(median_scores_df))), median_scores_df['median_score'], color='black')
plt.fill_between(
    list(range(len(median_scores_df))),
    median_scores_df['95% CI lower'],
    median_scores_df['95% CI upper'],
    color='lightgray',
    alpha=0.5
)
plt.xlabel('Hyperparameter combination')
plt.ylabel('Median balanced accuracy')
plt.show()

## 6. Cross-validation performance estimate

In [ ]:
if RUN_FULL_CV_ESTIMATE:
    estimate_folds = limited_folds(engineered_folds, ESTIMATE_FOLD_LIMIT)

    estimate_sampling = FoldSamplingConfig(
        use_sampling=ESTIMATE_USE_SAMPLING,
        train_sample_fraction=ESTIMATE_TRAIN_SAMPLE_FRAC,
        validation_sample_fraction=ESTIMATE_VALIDATION_SAMPLE_FRAC,
        sample_seed=RANDOM_SEARCH_SEED + 21,
    )

    estimate_eval_folds = make_fixed_sampled_folds(estimate_folds, estimate_sampling)

    full_fold_scores = score_xgb_on_folds_parallel(
        estimate_eval_folds,
        best_params,
        seed=RANDOM_SEARCH_SEED + 31,
        gpu_ids=PARALLEL_GPU_IDS,
        use_balanced_sample_weight=USE_BALANCED_CLASS_WEIGHT,
    )

    cv_results = summarize_scores(full_fold_scores)
    cv_results['params'] = best_params
    cv_results['fold_count_used'] = len(estimate_folds)
    cv_results['fold_count_available'] = len(engineered_folds)
    cv_results['use_sampling'] = ESTIMATE_USE_SAMPLING
    cv_results['use_balanced_class_weight'] = USE_BALANCED_CLASS_WEIGHT
    cv_results['train_sample_fraction'] = ESTIMATE_TRAIN_SAMPLE_FRAC
    cv_results['validation_sample_fraction'] = ESTIMATE_VALIDATION_SAMPLE_FRAC

    with open(CROSS_VALIDATION_SCORES, 'wb') as handle:
        pickle.dump(cv_results, handle)

else:
    with open(CROSS_VALIDATION_SCORES, 'rb') as handle:
        cv_results = pickle.load(handle)

ci_summary = summarize_with_ci(cv_results['fold_scores'])
cv_results['mean_ci_95'] = ci_summary['mean']
cv_results['median_ci_95'] = ci_summary['median']

print('Cross-validation summary')
print(
    f"Mean balanced accuracy:   {cv_results['mean_ci_95']['value']:.4f} "
    f"(95% CI: {cv_results['mean_ci_95']['ci_lower']:.4f}, {cv_results['mean_ci_95']['ci_upper']:.4f})"
)

print(
    f"Median balanced accuracy: {cv_results['median_ci_95']['value']:.4f} "
    f"(95% CI: {cv_results['median_ci_95']['ci_lower']:.4f}, {cv_results['median_ci_95']['ci_upper']:.4f})"
)

print(f"Std balanced accuracy:    {cv_results['std']:.4f}")
print(f"Folds used: {cv_results['fold_count_used']}/{cv_results['fold_count_available']}")
print(f"Used sampling: {cv_results['use_sampling']}")
print(f"Used balanced class weight: {cv_results.get('use_balanced_class_weight', False)}")

if cv_results['use_sampling']:
    print(f"Train sample fraction: {cv_results['train_sample_fraction']:.2f}")
    print(f"Validation sample fraction: {cv_results['validation_sample_fraction']:.2f}")

## 7. Train final model and generate submission

In [ ]:
x_train_full = train_df.drop('health_condition', axis=1)
y_train_full = train_df['health_condition']
x_test_full = test_df.drop('id', axis=1)

final_sample_weight = None
if USE_BALANCED_CLASS_WEIGHT:
    final_sample_weight = compute_sample_weight(class_weight='balanced', y=y_train_full.to_numpy())

final_model = build_xgb_model(best_params, seed=315, prefer_gpu=True)

try:
    final_model.fit(x_train_full, y_train_full, sample_weight=final_sample_weight)

except XGBoostError:
    final_model = build_xgb_model(best_params, seed=315, prefer_gpu=False)
    final_model.fit(x_train_full, y_train_full, sample_weight=final_sample_weight)

test_predictions = final_model.predict(x_test_full)

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'health_condition': label_encoder.inverse_transform(test_predictions.astype(int)),
})

submission_df.to_csv(FINAL_SUBMISSION_FILE, index=False)

print(f'Saved submission to {FINAL_SUBMISSION_FILE}')
print(submission_df['health_condition'].value_counts())
submission_df.head()